# NB03 — Volatility Modeling: Classical Econometrics

**Objectives**: Fit GARCH(1,1), GJR-GARCH, EGARCH, FIGARCH per ticker.
Compare innovation distributions (Normal, t, Skewed-t, GED). Model selection via AIC/BIC.

**FIGARCH skipped** for tickers with < 1500 obs (CRWD, DDOG, PLTR).

**Output**: `garch_parameters.csv`, `conditional_vol_series.parquet`

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.config import *
from src.feature_engineering import compute_log_returns
from src.garch_utils import *
from src.visualization import plot_conditional_volatility, save_fig
print('Imports OK')

## 1. Load Returns

In [ ]:
master = pd.read_parquet(MASTER_DATA_FILE)
adj_tickers = [t for t in TICKERS if t in master.columns]
log_ret = compute_log_returns(master[adj_tickers])
print(f'Returns: {log_ret.shape}')

## 1b. Long-Memory Detection: Hurst Exponent (Prerequisite for FIGARCH)

Mandatory check before fitting FIGARCH: estimate Hurst exponent (R/S) and
fractional differencing parameter d (GPH) on volatility proxies |r_t| and r²_t.

**Decision rule**: Only fit FIGARCH if H > 0.5 AND d > 0 with 95% CI excluding 0.

In [ ]:
from src.statistical_tests import hurst_rs, hurst_gph

hurst_results = {}
print(f"{'Ticker':>6s}  {'H(R/S)':>7s}  {'d(GPH)':>7s}  {'SE':>6s}  {'p-val':>7s}  {'FIGARCH?':>9s}")
print("-" * 55)

for t in adj_tickers:
    ret = log_ret[t].dropna().values
    h_rs = hurst_rs(np.abs(ret))
    d_gph, se_gph, p_gph = hurst_gph(ret ** 2)  # squared returns as vol proxy
    
    eligible = (h_rs > 0.5) and (d_gph > 0) and (p_gph < 0.05)
    # Also check minimum obs for FIGARCH
    if len(ret) < MIN_OBS_FIGARCH:
        eligible = False
        reason = f"<{MIN_OBS_FIGARCH} obs"
    elif not eligible:
        reason = "No long memory"
    else:
        reason = "Eligible"
    
    hurst_results[t] = {
        'H_RS': h_rs, 'd_GPH': d_gph, 'se_GPH': se_gph,
        'p_GPH': p_gph, 'figarch_eligible': eligible, 'reason': reason
    }
    print(f"{t:>6s}  {h_rs:7.3f}  {d_gph:7.3f}  {se_gph:6.3f}  {p_gph:7.4f}  {reason}")

hurst_df = pd.DataFrame(hurst_results).T
figarch_eligible = hurst_df[hurst_df['figarch_eligible'] == True].index.tolist()
print(f"\nFIGARCH-eligible tickers ({len(figarch_eligible)}): {figarch_eligible}")
hurst_df.to_csv(TABLES_DIR / 'nb03_hurst_exponent.csv')

## 2. Full GARCH Pipeline (20 Tickers × 4 Models × 4 Distributions)

In [ ]:
returns_dict = {t: log_ret[t].dropna() for t in adj_tickers}
params_table, cond_vol = run_full_garch_pipeline(returns_dict, save=True)
print(f'Parameters: {params_table.shape}')
print(f'Conditional vol: {cond_vol.shape}')

## 3. Best Model Per Ticker (by BIC)

In [ ]:
best_models = []
for t in adj_tickers:
    sub = params_table[params_table['ticker'] == t]
    if len(sub) == 0: continue
    best = sub.loc[sub['bic'].idxmin()]
    best_models.append({'ticker': t, 'model': best['model'], 'dist': best['distribution'],
                        'aic': best['aic'], 'bic': best['bic']})
pd.DataFrame(best_models).set_index('ticker')

## 4b. BH-FDR on ARCH-LM Tests & Realized Vol Signature Plot

In [ ]:
from src.statistical_tests import benjamini_hochberg
from statsmodels.stats.diagnostic import het_arch

# ARCH-LM test on standardized residuals for all tickers
arch_lm_results = []
for t in adj_tickers:
    sub = params_table[params_table['ticker'] == t]
    if len(sub) == 0:
        continue
    best = sub.loc[sub['bic'].idxmin()]
    spec = MODEL_SPECS.get(best['model'], dict(vol='GARCH', p=1, o=0, q=1))
    res = fit_garch(log_ret[t].dropna(), dist=best['distribution'], **spec)
    if res is not None:
        std_resid = res.std_resid
        try:
            lm_stat, lm_pval, _, _ = het_arch(std_resid, nlags=10)
            arch_lm_results.append({'ticker': t, 'lm_stat': lm_stat, 'lm_pvalue': lm_pval})
        except Exception:
            pass

arch_lm_df = pd.DataFrame(arch_lm_results)
if len(arch_lm_df) > 0:
    raw_pvals = arch_lm_df['lm_pvalue'].values
    rejected, adjusted = benjamini_hochberg(raw_pvals, q=0.05)
    arch_lm_df['lm_pvalue_bh'] = adjusted
    arch_lm_df['reject_bh'] = rejected
    print("--- ARCH-LM Tests (BH-FDR Corrected) ---")
    print(f"Remaining ARCH effects after best GARCH fit: {sum(rejected)}/{len(rejected)} tickers")
    print(arch_lm_df[['ticker', 'lm_stat', 'lm_pvalue', 'lm_pvalue_bh', 'reject_bh']])

# Realized Volatility Signature Plot (5 most liquid tickers)
print("\n--- Realized Vol Signature Plot ---")
print("Validates that daily OHLC-based estimators are appropriate.")
print("(Requires intraday data — will use available daily frequencies as proxy)")
from openbb import obb

liquid_tickers = ['NVDA', 'AAPL', 'MSFT', 'META', 'AMZN']
fig, axes = plt.subplots(1, len(liquid_tickers), figsize=(20, 4))
for ax, t in zip(axes, liquid_tickers):
    try:
        result = obb.equity.price.historical(
            symbol=t, start_date=(pd.Timestamp.now() - pd.Timedelta(days=60)).strftime('%Y-%m-%d'),
            end_date=pd.Timestamp.now().strftime('%Y-%m-%d'),
            interval='5m',
        )
        intraday = result.to_df()
        col_map = {"open": "Open", "high": "High", "low": "Low",
                    "close": "Close", "volume": "Volume"}
        intraday.rename(columns=col_map, inplace=True)
        if len(intraday) > 0:
            # Compute RV at different sampling frequencies
            log_ret_5m = np.log(intraday['Close'] / intraday['Close'].shift(1)).dropna()
            freqs = {'5min': 1, '15min': 3, '30min': 6, '1hr': 12}
            rv_points = {}
            for label, n_bars in freqs.items():
                sampled = log_ret_5m.iloc[::n_bars]
                rv_points[label] = (sampled ** 2).sum() * 252 / len(intraday) * len(sampled)
            ax.bar(rv_points.keys(), rv_points.values(), alpha=0.7)
            ax.set_title(t, fontsize=10)
            ax.set_ylabel('Annualized RV')
    except Exception as e:
        ax.set_title(f'{t} (unavail)')
fig.suptitle('Realized Volatility Signature Plot (60d intraday)', fontsize=12)
fig.tight_layout()
save_fig(fig, 'nb03_rv_signature_plot')
plt.show()

## 4. Residual Diagnostics

In [ ]:
diag_rows = []
for t in adj_tickers[:5]:
    sub = params_table[params_table['ticker'] == t]
    best = sub.loc[sub['bic'].idxmin()]
    spec = MODEL_SPECS.get(best['model'], dict(vol='GARCH', p=1, o=0, q=1))
    res = fit_garch(log_ret[t].dropna(), dist=best['distribution'], **spec)
    if res is not None:
        diag = residual_diagnostics(res)
        diag['ticker'] = t
        diag_rows.append(diag)
pd.DataFrame(diag_rows).set_index('ticker')

## 5. Conditional Volatility Plot

In [ ]:
fig = plot_conditional_volatility(cond_vol, adj_tickers[:6],
    title='GARCH Conditional Volatility (Top 6)', save_name='nb03_cond_vol')
plt.show()